# Create Table for Paper with 
Age, gender
IQ screener (verbal + visuo-spatial)
Visuo-spatial short-term memory (Corsi)
Standardized math achievement (e.g., DEMAT)
Math anxiety / confidence questionnaires

- code from numrisk/behavior_general/age_iq_effects

In [30]:
import pandas as pd
import numpy as np
import os.path as op
import seaborn as sns
import pingouin
import matplotlib.pyplot as plt
from scipy import stats

params_folder = '/Users/mrenke/data/ds-dnumrisk/derivatives/phenotype'
bids_folder = '/Users/mrenke/data/ds-dnumrisk'

group_list =  pd.read_csv(op.join(bids_folder, 'group_mapping.csv'), header=0, index_col=0)

In [ ]:
# Age & Gender
df_part = pd.read_csv(op.join('/Users/mrenke/data/ds-dnumrisk/add_tables','subjects_recruit_scan_scanned-final.csv'), header=0) #, index_col=0
df_part = df_part.loc[:,['subject ID', 'age','group','gender']].rename(mapper={'subject ID': 'subject'},axis=1).dropna().astype({'subject': int, 'group': int}).set_index('subject')
df_part['group'] = np.where(df_part['group'] == 0, 'control', 'dyscalc')

In [23]:
# Math skill, confidence, anxiety
math_stuff = pd.read_csv(op.join(params_folder, 'math_skill&confidence&anxiety-means.csv')).set_index('subject')

# Corsi
vs_wm = pd.read_csv(op.join(params_folder, 'visio-spatial-WM_CBTtask-params.csv'))
vs_wm.set_index('subject', inplace=True)

# Weber Fraction
pana = pd.read_csv('/Users/mrenke/data/ds-dnumrisk/add_tables/panamath_AllRunsSummary.csv').rename(mapper={'Subject ID': 'subject'}, axis=1).set_index('subject')['Weber Fraction'] #[['Number of Trials','Weber Fraction', 'Percent Correct', 'Number of Non-RT-Outlier Trials']]

# IQ - verbal & visuo-spatial reasoning (matrices..)
iq_scores = pd.read_csv(op.join(params_folder, 'iq-scores_ids2.csv')).set_index('subject')

# combine
df_comb = df_part.join([math_stuff, vs_wm, pana, iq_scores], how='inner')

In [24]:
# check if joing on subject worked
df_comb.loc[46].to_frame()

,46
age,23.0
group,dyscalc
gender,w
skill_score,5.0
anx_mean,4.0
conf_mean,2.0
basisscore,5
overall_score,9.0
erfassungsspanne,6
Weber Fraction,0.243674


In [25]:
var_list = ['group', 'age', 'gender', 'me', 'kn','erfassungsspanne','skill_score','anx_mean', 'conf_mean', 'Weber Fraction']
name_list = ['Age', 'Gender', 'Visuospatial reasoning', 'Verbal reasoning', 'Visuospatial working memory', 'Math skills', 'Math anxiety', 'Math confidence', 'Weber Fraction']
point_list = ['Age', 'Gender', 'IQ points','IQ points','Corsi span', 'Test points','Anxiety rating', 'Confidence rating', 'Weber Fraction']

In [27]:
df_comb[var_list]

,group,age,gender,me,kn,erfassungsspanne,skill_score,anx_mean,conf_mean,Weber Fraction
subject,,,,,,,,,,
1,control,19.0,w,120.0,105.0,8,36.0,1.333333,2.666667,0.143232
2,dyscalc,17.0,w,100.0,95.0,6,6.0,2.666667,0.666667,0.119873
3,control,15.0,w,115.0,95.0,5,29.0,1.666667,3.333333,0.349204
4,dyscalc,17.0,w,110.0,105.0,6,34.0,4.000000,1.833333,0.183717
5,control,20.0,w,90.0,115.0,7,30.0,2.000000,2.333333,0.154192
...,...,...,...,...,...,...,...,...,...,...
62,control,23.0,w,125.0,115.0,6,28.0,2.666667,2.666667,0.188153
63,control,23.0,m,115.0,115.0,8,42.0,1.000000,3.666667,0.140090
64,control,23.0,m,75.0,120.0,7,50.0,1.666667,3.666667,0.156378


In [29]:
vars_continuous = ['age', 'me', 'kn','erfassungsspanne','skill_score','anx_mean', 'conf_mean', 'Weber Fraction']

In [67]:
def mean_sd(x):
    return f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

def format_p(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"



In [68]:
rows = []

for var in vars_continuous:
    ctrl = df_comb[df_comb["group"] == "control"][var].dropna()
    dys  = df_comb[df_comb["group"] == "dyscalc"][var].dropna()

    # Welch t-test
    t, p = stats.ttest_ind(ctrl, dys, equal_var=False)

    rows.append({
        "Measure": var,
        "Control (mean ± SD)": mean_sd(ctrl),
        "Dyscalculia (mean ± SD)": mean_sd(dys),
        "t": round(t, 2),
        "p": format_p(p)
    })

table1 = pd.DataFrame(rows)


In [69]:
gender_table = (
    df_comb
    .groupby(["group", "gender"])
    .size()
    .unstack(fill_value=0)
)

# Chi-square test
chi2, p_gender, _, _ = stats.chi2_contingency(gender_table)

gender_summary = pd.DataFrame({
    "Measure": ["Gender (f / m)"],
    "Control (mean ± SD)": [f"{gender_table.loc['control', 'w']} / {gender_table.loc['control', 'm']}"],
    "Dyscalculia (mean ± SD)": [f"{gender_table.loc['dyscalc', 'w']} / {gender_table.loc['dyscalc', 'm']}"],
    "t": ["χ²"],
    "p": [round(p_gender, 3)]
})


In [70]:
table1 = pd.concat([gender_summary, table1], ignore_index=True)
label_map = {
    "age": "Age (years)",
    "kn": "IQ screener (verbal)",
    "me": "IQ screener (visuo-spatial)",
    "erfassungsspanne": "Visuo-spatial short-term memory (Corsi span)",
    "skill_score": "Mathematical achievement (DEMAT test)",
    "anx_mean": "Math anxiety",
    "conf_mean": "Math confidence",
    "Weber Fraction": "Panamath (Weber fraction)"
}

table1["Measure"] = table1["Measure"].replace(label_map)


In [71]:
print(table1.to_string(index=False))


                                     Measure Control (mean ± SD) Dyscalculia (mean ± SD)     t       p
                              Gender (f / m)              28 / 5                  28 / 5    χ²     1.0
                                 Age (years)        19.18 ± 2.86            19.09 ± 2.71  0.13   0.895
                 IQ screener (visuo-spatial)      102.58 ± 15.87           93.79 ± 12.93  2.47   0.016
                        IQ screener (verbal)      107.27 ± 12.13          103.03 ± 12.05  1.43   0.159
Visuo-spatial short-term memory (Corsi span)         7.48 ± 1.20             6.36 ± 0.86  4.36 < 0.001
       Mathematical achievement (DEMAT test)        33.30 ± 9.50            17.88 ± 8.93   6.8 < 0.001
                                Math anxiety         2.34 ± 0.80             3.73 ± 0.89 -6.65 < 0.001
                             Math confidence         3.15 ± 0.69             1.65 ± 0.81  8.09 < 0.001
                   Panamath (Weber fraction)         0.18 ± 0.06         

In [72]:
fn_out = '/Users/mrenke/Desktop/DNumRisk/paper/'
table1.to_csv(fn_out + "Table1_participant_characteristics.csv", index=False)
